In [12]:
#creating consumer, pulls data
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'green-trips'

In [13]:
#deserializer
import json
import sys, os
sys.path.append(os.path.abspath("../src"))
from models import Ride, ride_deserializer

In [14]:
#
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-to-postgres',
    #value_deserializer=lambda x: json.loads(x)
    value_deserializer=ride_deserializer
)

In [15]:
#connecting to postgres
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [ ]:
#Read messages and insert into PostgreSQL:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(int(ride.lpep_pickup_datetime) / 1000)
    dropoff_dt = datetime.fromtimestamp(int(ride.lpep_dropoff_datetime) / 1000)
    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime, dropoff_datetime, passenger_count, tip_amount)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID,
         ride.trip_distance, ride.total_amount, pickup_dt, dropoff_dt,
         ride.passenger_count, ride.tip_amount)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()